# Training

## Setup

In [1]:
import os, sys, subprocess, shutil

subprocess.run(["pip", "install", "-q", "segmentation-models-pytorch"])

repo_dir = "/kaggle/working/repo"
if os.path.exists(repo_dir):
    shutil.rmtree(repo_dir)
!git clone https://github.com/TigranBoyakhchyan/GeoSpill-AI.git {repo_dir}

sys.path.insert(0, f"{repo_dir}/src")
print("Setup complete")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 92.4 MB/s eta 0:00:00
Cloning into '/kaggle/working/repo'...


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

remote: Enumerating objects: 133, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (87/87), done.
remote: Total 133 (delta 47), reused 120 (delta 35), pack-reused 0 (from 0)
Receiving objects: 100% (133/133), 20.41 MiB | 48.95 MiB/s, done.
Resolving deltas: 100% (47/47), done.
Setup complete


## Fix paths in train_stats.json

In [2]:
import json

INPUT_BASE = "/kaggle/input/datasets/tigranboyakhchyan05/sar-oil-spill-preprocessed/results/output"
STATS_SRC  = "/kaggle/input/datasets/tigranboyakhchyan05/sar-oil-spill-preprocessed/train_stats.json"

with open(STATS_SRC) as f:
    stats = json.load(f)

# Show what the old paths look like
print("Old paths (first 2):")
for p in stats["splits"]["train"][:2]:
    print(f"  {p}")

# Find the old prefix by looking at the first path
old_path = stats["splits"]["train"][0]
# Everything before /npz_cache/ is the old prefix
old_prefix = old_path.split("/npz_cache/")[0]
new_prefix = f"{INPUT_BASE}/npz_cache"

print(f"\nReplacing: {old_prefix}/npz_cache")
print(f"     With: {new_prefix}")

def fix(p):
    return p.replace(f"{old_prefix}/npz_cache", new_prefix)

stats["splits"]["train"] = [fix(p) for p in stats["splits"]["train"]]
stats["splits"]["val"]   = [fix(p) for p in stats["splits"]["val"]]
stats["splits"]["test"]  = [fix(p) for p in stats["splits"]["test"]]
stats["splits"]["masks"] = {fix(k): fix(v) for k, v in stats["splits"]["masks"].items()}

# Save fixed version to writable directory
STATS_FILE = "/kaggle/working/train_stats.json"
with open(STATS_FILE, "w") as f:
    json.dump(stats, f, indent=2)

# Verify
print("\nNew paths (first 2):")
for p in stats["splits"]["train"][:2]:
    exists = os.path.exists(p)
    print(f"  {p} — {'OK' if exists else 'MISSING'}")

print(f"\nSplit sizes: {stats['n_train']} train / {stats['n_val']} val / {stats['n_test']} test")

Old paths (first 2):
  /kaggle/working/output/npz_cache/images/01315.npz
  /kaggle/working/output/npz_cache/images/00974.npz

Replacing: /kaggle/working/output/npz_cache
     With: /kaggle/input/datasets/tigranboyakhchyan05/sar-oil-spill-preprocessed/results/output/npz_cache

New paths (first 2):
  /kaggle/input/datasets/tigranboyakhchyan05/sar-oil-spill-preprocessed/results/output/npz_cache/images/01315.npz — OK
  /kaggle/input/datasets/tigranboyakhchyan05/sar-oil-spill-preprocessed/results/output/npz_cache/images/00974.npz — OK

Split sizes: 840 train / 180 val / 180 test


## Dataset sanity check

In [3]:
from dataset import build_datasets

STATS_FILE = "/kaggle/working/train_stats.json"
datasets = build_datasets(STATS_FILE, patch_size=256)

for split, ds in datasets.items():
    print(f"{split:5s}: {len(ds)} samples")

img, msk = datasets["train"][0]
print(f"\nSample — image: {img.shape}, dtype: {img.dtype}")
print(f"  range: [{img.min():.2f}, {img.max():.2f}]")
print(f"  mask: {msk.shape}, unique: {msk.unique().tolist()}")

train: 840 samples
val  : 180 samples
test : 180 samples

Sample — image: torch.Size([2, 256, 256]), dtype: torch.float32
  range: [-4.82, 5.44]
  mask: torch.Size([1, 256, 256]), unique: [0.0]


## Train

In [4]:
from types import SimpleNamespace
import training

args = SimpleNamespace(
    stats_file    = "/kaggle/working/train_stats.json",
    output_dir    = "/kaggle/working/results",
    resume        = None,
    epochs        = 50,
    batch_size    = 8,
    lr            = 1e-4,
    warmup_epochs = 5,
    patch_size    = 256,
    num_workers   = 2,
)

training.train(args)

Loading datasets...
  Train: 840 samples  Val: 180 samples  Test: 180 samples


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

  SAR Oil Spill Detection — Training
  Device       : cuda
  Architecture : U-Net + ResNet34 (pretrained ImageNet)
  Loss         : Focal + Dice
  Trainable params: 24,433,233
  Epochs       : 1 -> 50
  Batch size   : 8
  Patch size   : 256
  LR           : 0.0001
  Stats file   : /kaggle/working/train_stats.json
  Output dir   : /kaggle/working/results
[001/50] loss 1.0516 -> 1.0089 | IoU 0.1576 | Dice 0.2724 | Prec 0.1619 | Rec 0.8569 | LR 4.00e-05 | 102s
  New best — IoU 0.1576, Dice 0.2724 -> saved to /kaggle/working/results/best_model_20260604_091953.pt
[002/50] loss 0.9907 -> 0.9450 | IoU 0.5824 | Dice 0.7361 | Prec 0.6162 | Rec 0.9140 | LR 6.00e-05 | 84s
  New best — IoU 0.5824, Dice 0.7361 -> saved to /kaggle/working/results/best_model_20260604_091953.pt
[003/50] loss 0.9649 -> 0.9679 | IoU 0.5976 | Dice 0.7481 | Prec 0.6190 | Rec 0.9453 | LR 8.00e-05 | 83s
  New best — IoU 0.5976, Dice 0.7481 -> saved to /kaggle/working/results/best_model_20260604_091953.pt
[004/50] loss 0.944